# Trained-model results and plots
After copying the full Computerome and Gefion exports, choose **Restart Kernel and Run All Cells**. This notebook reuses the existing bootstrap engine and latent-analysis tools. It does not train models.

Performance summaries can supply performance plots. **t-SNE additionally needs exported latent vectors and measured participant metadata, or saved t-SNE coordinates.** Missing predictions, intervals and latent inputs remain visible in the inventories. Total **peripheral** plaque volume sums the four carotid/femoral sides; coronary volume is separate.

Keep the notebook inside `paper_analysis/` in the full repository. Plotting requires Python, pandas, numpy, matplotlib and scikit-learn. No mesh archive or GPU is needed for exported latent CSVs.

In [ ]:
from pathlib import Path
import hashlib, json, re, sys, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
REPO = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / 'paper_analysis/collect_all_results.py').is_file()), None)
if REPO is None:
    raise FileNotFoundError('Open this notebook from the repository root or paper_analysis folder.')
sys.path.insert(0, str(REPO))
from paper_analysis.bootstrap_metrics import load_predictions, summarize_model
from paper_analysis.analyze_latent_signal import standardize_latent, choose_kmeans, make_tsne_figure
from paper_analysis.run_modality_benchmark import load_latent, load_metadata, merge_split
from paper_analysis.data_utils import read_table, numeric, resolve_id_column, clean_participant_id
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})

## 1. Inputs
Copy each **entire export directory**, including `plot_data/` and `plot_artifacts.csv`. Use the two default server folders below, or a single merged export. Missing exports are listed explicitly. Set `INCLUDE_PROVISIONAL_REFERENCE=True` to display the existing 46-row presentation transcription when neither server export exists; it cannot produce CIs or t-SNE.

`REMOTE_TO_LOCAL` optionally maps old server paths to local copies. Latent jobs are discovered from bundled `latent_manifest.json` files. Add explicit `LATENT_JOBS` to specify selected models and their input description.

In [ ]:
EXPORT_DIRS = [REPO / 'outputs/model_overview/results_overview_computerome', REPO / 'outputs/model_overview/results_overview_gefion']
# Alternatively: EXPORT_DIRS = [REPO / 'outputs/model_overview/combined']
OUTPUT = REPO / 'outputs/model_overview/plots'
OUTPUT.mkdir(parents=True, exist_ok=True)
REMOTE_TO_LOCAL = {}  # {'/remote/checkpoints': REPO / 'downloaded/checkpoints'}
INCLUDE_PROVISIONAL_REFERENCE = False
COMPUTE_MISSING_CIS = True  # Explicit held-out participant predictions only; cached.
BOOTSTRAPS, CONFIDENCE, SEED = 5000, .95, 20261007
PLOT_METRICS = ['auc', 'average_precision', 'r2', 'mae', 'spearman']
MAX_ROWS_PER_FIGURE = 30
RUN_TSNE, PERPLEXITY = True, 30
LATENT_JOBS = []
# Optional explicit job, with local paths or remote paths resolved above:
# LATENT_JOBS = [dict(name='EXACT_EXPERIMENT', gender='F',
#     representation_label='Scan + age + weight; posterior mean',
#     train_latent='/path/train_latent_embeddings.csv', test_latent='/path/test_latent_embeddings.csv',
#     train_tsv='/path/train_enriched.tsv', test_tsv='/path/test_enriched.tsv',
#     target_col='anyplaq', volume_col='plaque_volume_total_corrected')]
# volume_col must be measured metadata, never predictions.

In [ ]:
def read_csv(path):
    try: return pd.read_csv(path, dtype=str, keep_default_na=False)
    except pd.errors.EmptyDataError: return pd.DataFrame()

keys = ['all_results', 'confidence_intervals', 'result_inventory', 'plot_artifacts']
tables = {key: [] for key in keys}
input_status = []
for directory in EXPORT_DIRS:
    directory = Path(directory).expanduser().resolve()
    if not (directory / 'all_results.csv').is_file():
        input_status.append(dict(export=str(directory), status='missing', rows=0)); continue
    for key in keys:
        if (directory / f'{key}.csv').is_file():
            frame = read_csv(directory / f'{key}.csv'); frame['_export_dir'] = str(directory)
            if key == 'plot_artifacts':
                # A merged index points into its original server export directories.
                frame['_artifact_dir'] = [str(Path(value).parent) if value and Path(value).is_file() else str(directory) for value in frame.get('export_file', pd.Series('', index=frame.index))]
            tables[key].append(frame)
    input_status.append(dict(export=str(directory), status='loaded', rows=len(tables['all_results'][-1])))
if not tables['all_results'] and INCLUDE_PROVISIONAL_REFERENCE:
    directory = REPO / 'outputs/model_overview/provisional_gefion'
    if (directory / 'all_results.csv').is_file():
        frame = read_csv(directory / 'all_results.csv'); frame['_export_dir'] = str(directory)
        tables['all_results'].append(frame)
        input_status.append(dict(export=str(directory), status='provisional transcription only', rows=len(frame)))
results, saved_ci, inventory, artifacts = [pd.concat(tables[k], ignore_index=True).fillna('') if tables[k] else pd.DataFrame() for k in keys]
for field in ['platform', 'source_root', 'target', 'cohort', 'gender', 'seed', 'prior', 'task', 'target_transform', 'model', 'run', 'run_directory', 'evaluation_split', 'result_scope', 'evidence_level', 'preferred', 'source_file', 'source_row', 'reference_run', 'reference_fingerprint', 'reference_split_hashes']:
    if field not in results: results[field] = ''
results['_row_id'] = [f'{i:06d}' for i in range(len(results))]
preferred = results['preferred'].astype(str).str.lower().isin(['true', '1'])
provisional = results['evidence_level'].eq('transcribed_screenshot')
selected = results.loc[preferred | (provisional & INCLUDE_PROVISIONAL_REFERENCE)].copy()
display(pd.DataFrame(input_status))
print(f'{len(results)} saved rows; {len(selected)} preferred/reference rows; {len(saved_ci)} saved CI rows; {len(artifacts)} artifacts.')
if not inventory.empty:
    display(inventory.loc[~inventory.get('status', pd.Series('', index=inventory.index)).eq('collected')].head(100))
results.to_csv(OUTPUT / 'all_results_with_provenance.csv', index=False)
inventory.to_csv(OUTPUT / 'source_inventory.csv', index=False)

## 2. Metrics and confidence intervals
Keep target, cohort, sex, platform, checkpoint root, evaluation split and metric scale separate. Participant intervals and training-seed intervals answer different questions. Standard deviations are **not** treated as CIs.

Missing participant CIs are calculated with the existing `load_predictions` and `summarize_model` only for preferred results with explicit held-out provenance and an available prediction artifact. Study baseline columns are scored separately on the same held-out participants. Existing 95% CIs are linked only by the collector's specific `ci_files` and matching model/metric. Unknown or ambiguous intervals remain in `saved_ci_unlinked.csv`.

In [ ]:
ALIASES = {
    'auc': ['auc', 'roc_auc', 'test_auc', 'test.roc_auc'],
    'average_precision': ['averageprecision', 'average_precision', 'ap', 'test_average_precision', 'test_ap', 'test.average_precision'],
    'r2': ['r2', 'test_r2', 'test.r2'], 'mae': ['mae', 'test_mae', 'test.mae'],
    'rmse': ['rmse', 'test_rmse'], 'spearman': ['spearman', 'rho', 'test_spearman'],
    'pearson': ['pearson', 'test_pearson'], 'balanced_accuracy': ['balancedaccuracy', 'balanced_accuracy', 'test_balanced_accuracy'],
    'accuracy': ['accuracy', 'test_accuracy'], 'f1': ['f1', 'test_f1'],
    'sensitivity': ['sensitivity', 'test_sensitivity'], 'specificity': ['specificity', 'test_specificity'], 'brier': ['brier'],
    'transformed_r2': ['transformedr2', 'transformed_r2'], 'transformed_mae': ['transformedmae', 'transformed_mae'],
    'transformed_rmse': ['transformedrmse', 'transformed_rmse'],
    'transformed_pearson': ['transformedpearson', 'transformed_pearson'], 'transformed_spearman': ['transformedspearman', 'transformed_spearman']}
def metric_name(value):
    value = str(value).strip().lower()
    return next((name for name, aliases in ALIASES.items() if value in aliases), value)
def scalar(value):
    try: return float(value)
    except (TypeError, ValueError): return np.nan
def normalized_path(value): return str(value).replace('\\', '/').rstrip('/')
def artifact_path(row): return Path(row.get('_artifact_dir', row['_export_dir'])) / row['exported_file']
def local_artifact(original, export_dir=None):
    original = normalized_path(original)
    if not artifacts.empty and 'source_file' in artifacts:
        found = artifacts.loc[artifacts['source_file'].map(normalized_path).eq(original)]
        if export_dir is not None: found = found.loc[found['_export_dir'].eq(str(export_dir))]
        paths = {artifact_path(row) for _, row in found.iterrows()}
        paths = [path for path in paths if path.is_file()]
        if len(paths) == 1: return paths[0]
    for remote, local in sorted(REMOTE_TO_LOCAL.items(), key=lambda item: -len(item[0])):
        prefix = normalized_path(remote)
        if original == prefix or original.startswith(prefix + '/'):
            path = Path(local) / original[len(prefix):].lstrip('/')
            if path.is_file(): return path
    path = Path(original).expanduser()
    return path if path.is_file() else None

metric_rows = []
for _, row in selected.iterrows():
    lower_keys = {str(key).lower(): key for key in row.index}
    metrics = {}
    for metric, aliases in ALIASES.items():
        names = [alias + '_mean' for alias in aliases] if row['result_scope'] == 'seed_aggregate' else aliases
        values = [(scalar(row[lower_keys[name]]), lower_keys[name]) for name in names if name in lower_keys and np.isfinite(scalar(row[lower_keys[name]]))]
        if values: metrics[metric] = values[0]
    if row['result_scope'] == 'seed_paired_effect':
        metrics = {metric_name(row.get('Metric', '')): (scalar(row.get('MeanDirectionalImprovement')), 'MeanDirectionalImprovement')}
    for metric, (estimate, column) in metrics.items():
        if not np.isfinite(estimate): continue
        scale = 'transformed' if metric.startswith('transformed_') else ('original_target_units' if row['task'] in ['regression', 'hurdle'] else 'dimensionless')
        item = {**row.to_dict(), 'metric': metric, 'estimate': estimate, 'metric_scale': scale, 'original_metric_column': column,
                'ci_low': np.nan, 'ci_high': np.nan, 'confidence_level': np.nan, 'ci_status': 'missing', 'ci_source': '', 'interval_method': '',
                'resampling_unit': 'seed' if row['result_scope'].startswith('seed_') else 'participant',
                'n_participants': scalar(row.get('N', row.get('n', row.get('n_participants', '')))),
                'n_bootstrap_requested': np.nan, 'n_bootstrap_valid': np.nan}
        if row['result_scope'] == 'seed_paired_effect':
            item.update(ci_low=scalar(row.get('Improvement95CI_Low')), ci_high=scalar(row.get('Improvement95CI_High')),
                        confidence_level=.95, ci_status='saved_seed_interval', ci_source=row['source_file'], interval_method='paired-seed bootstrap; directional improvement')
        metric_rows.append(item)
tidy = pd.DataFrame(metric_rows)
linked_intervals = set()
if not tidy.empty and not saved_ci.empty:
    for index, row in tidy.iterrows():
        try: ci_files = json.loads(row.get('ci_files') or '[]')
        except (json.JSONDecodeError, TypeError): ci_files = []
        if not ci_files or row['result_scope'] != 'model_evaluation': continue
        matches = saved_ci.loc[saved_ci['source_file'].isin(ci_files)].copy()
        if 'metric' not in matches: continue
        matches = matches.loc[matches['metric'].map(metric_name).eq(row['metric']) & matches['model'].eq(row['model'])]
        matches = matches.loc[matches.get('resampling_unit', pd.Series('', index=matches.index)).str.lower().eq('participant')]
        matches = matches.loc[~matches['model'].str.contains('minus', case=False)]
        bounds = {(scalar(ci.get('ci_low')), scalar(ci.get('ci_high'))) for _, ci in matches.iterrows()}
        bounds = {(lo, hi) for lo, hi in bounds if np.isfinite(lo) and np.isfinite(hi) and lo <= hi}
        if len(bounds) != 1: continue
        ci = matches.iloc[0]; level = scalar(ci.get('confidence_level', ci.get('confidence', '')))
        if not np.isfinite(level) and '95' in str(ci.get('ci_source_field', '')): level = .95
        if not np.isclose(level, CONFIDENCE): continue
        lo, hi = next(iter(bounds))
        tidy.loc[index, ['ci_low', 'ci_high', 'confidence_level', 'ci_status', 'ci_source', 'interval_method']] = [lo, hi, level, 'saved_interval', ci['source_file'], ci.get('interval_method', 'saved participant bootstrap')]
        linked_intervals.update(matches.index)
ci_audit = []

In [ ]:
def prediction_for(row):
    source = normalized_path(row['source_file']); folder = source.rsplit('/', 1)[0]; candidates = []
    if '/study/heldout/' in source: candidates.append(folder + '/predictions.csv')
    for field in ['ResultsDir', 'results_dir']:
        value = row.get(field, '')
        if value:
            value = normalized_path(value)
            if not value.startswith('/') and not re.match(r'^[A-Za-z]:/', value): value = normalized_path(row['run_directory']) + '/' + value
            candidates.append(value + '/predictions.csv')
    candidates.extend([folder + '/test_predictions.csv', folder + '/predictions.csv'])
    for original in dict.fromkeys(candidates):
        path = local_artifact(original, row['_export_dir'])
        if path: return original, path
    return '', None

cache = OUTPUT / 'ci_cache'; cache.mkdir(exist_ok=True)
if COMPUTE_MISSING_CIS and not tidy.empty:
    for row_id, group in tidy.groupby('_row_id', sort=False):
        row = group.iloc[0]
        if row['result_scope'] != 'model_evaluation' or row['evidence_level'] == 'transcribed_screenshot' or group['ci_low'].notna().all(): continue
        entry = {'row_id': row_id, 'run': row['run'], 'model': row['model'], 'source_file': row['source_file']}
        original, path = prediction_for(row)
        if path is None:
            ci_audit.append({**entry, 'status': 'missing_predictions', 'reason': 'Collect with --include-plot-data or map server paths.'}); continue
        try:
            source_frame = pd.read_csv(path, dtype=str)
            explicit_test_rows = 'split' in source_frame and source_frame['split'].astype(str).str.lower().isin(['test', 'heldout', 'held_out', 'held-out']).any()
            if str(row['evaluation_split']).lower() not in ['test', 'heldout', 'held_out', 'held-out'] and not explicit_test_rows:
                ci_audit.append({**entry, 'status': 'unverified_split', 'reason': 'Requires explicit held-out provenance.'}); continue
            if explicit_test_rows:
                tidy.loc[tidy['_row_id'].eq(row_id), 'evaluation_split'] = 'test'
            task = 'regression' if row['task'] == 'hurdle' else row['task']
            if task not in ['binary', 'regression']: raise ValueError(f'Unknown task {task!r}.')
            signature = hashlib.sha256(path.read_bytes() + f'{row["model"]}|{row["target"]}|{task}|{BOOTSTRAPS}|{CONFIDENCE}|{SEED}'.encode()).hexdigest()[:20]
            cached = cache / f'{signature}.csv'
            if cached.is_file(): intervals = pd.read_csv(cached)
            else:
                load_path = path
                if row['model'] in source_frame.columns and 'target' in source_frame:
                    fields = {'participant_id': source_frame['participant_id'], 'target': source_frame['target'],
                              'probability' if task == 'binary' else 'prediction': source_frame[row['model']]}
                    if 'split' in source_frame: fields['split'] = source_frame['split']
                    load_path = cache / f'{signature}_predictions.csv'; pd.DataFrame(fields).to_csv(load_path, index=False)
                frame, detected = load_predictions(load_path, task=task, threshold=.5, duplicate_policy='error', require_participant_id=True)
                intervals = summarize_model(row['model'], frame, task, BOOTSTRAPS, SEED, CONFIDENCE, regression_strata='auto')
                intervals.to_csv(cached, index=False)
            for _, ci in intervals.iterrows():
                mask = tidy['_row_id'].eq(row_id) & tidy['metric'].eq(metric_name(ci['metric']))
                tidy.loc[mask, ['estimate', 'ci_low', 'ci_high', 'confidence_level', 'ci_status', 'ci_source', 'interval_method', 'n_participants', 'n_bootstrap_requested', 'n_bootstrap_valid']] = [ci['estimate'], ci['ci_low'], ci['ci_high'], ci['confidence_level'], 'participant_bootstrap', original, ci['interval_method'], ci['n_participants'], ci['n_bootstrap_requested'], ci['n_bootstrap_valid']]
            ci_audit.append({**entry, 'status': 'completed', 'prediction_file': original, 'cache': str(cached), 'reason': ''})
            print(f'CI: {row["platform"]} / {row["run"]} / {row["model"]}')
        except Exception as error:
            ci_audit.append({**entry, 'status': 'failed', 'prediction_file': original, 'reason': f'{type(error).__name__}: {error}'})
if not tidy.empty:
    tidy['has_95ci'] = (np.isclose(pd.to_numeric(tidy['confidence_level'], errors='coerce'), .95)
        & np.isfinite(pd.to_numeric(tidy['ci_low'], errors='coerce')) & np.isfinite(pd.to_numeric(tidy['ci_high'], errors='coerce')) & tidy['ci_low'].le(tidy['ci_high']))
    tidy.to_csv(OUTPUT / 'tidy_metrics.csv', index=False)
    tidy.loc[~tidy['has_95ci']].to_csv(OUTPUT / 'missing_confidence_intervals.csv', index=False)
    display(tidy.groupby(['platform', 'evidence_level', 'resampling_unit', 'has_95ci'], dropna=False).size().rename('metric_rows').reset_index())
pd.DataFrame(ci_audit, columns=sorted(set().union(*(row.keys() for row in ci_audit))) if ci_audit else ['row_id', 'run', 'model', 'status', 'reason']).to_csv(OUTPUT / 'ci_computation_inventory.csv', index=False)
if not saved_ci.empty: saved_ci.loc[~saved_ci.index.isin(linked_intervals)].to_csv(OUTPUT / 'saved_ci_unlinked.csv', index=False)

## 3. Performance plots
Each panel contains one target/cohort/sex/root/split/scale. Runs are ordered by name, not test score. Filled circles and lines show actual 95% CIs; hollow orange circles explicitly mark missing intervals. Screenshot results have their own provisional panels.

MAE/RMSE retain the source target units; physical volume units cannot be determined from a summary export alone. R², correlations and discrimination metrics are dimensionless. Positive-only and upper-tail cohorts remain separate from the whole cohort.

In [ ]:
plot_manifest = []
if not tidy.empty:
    rows = tidy.loc[tidy['metric'].isin(PLOT_METRICS)].copy()
    group_fields = ['platform', 'source_root', 'target', 'cohort', 'gender', 'evaluation_split', 'metric_scale', 'result_scope', 'evidence_level', 'resampling_unit', 'metric']
    for field in group_fields: rows[field] = rows[field].replace('', 'unspecified')
    for key, group in rows.groupby(group_fields, dropna=False, sort=True):
        group = group.sort_values(['run', 'model', 'seed', 'source_file', 'source_row'], kind='stable'); context = dict(zip(group_fields, key))
        for page, start in enumerate(range(0, len(group), MAX_ROWS_PER_FIGURE), 1):
            chunk = group.iloc[start:start + MAX_ROWS_PER_FIGURE]
            fig, ax = plt.subplots(figsize=(11, max(3.5, .52 * len(chunk) + 2))); ys = np.arange(len(chunk))
            for y, (_, row) in zip(ys, chunk.iterrows()):
                if row['has_95ci']:
                    ax.hlines(y, row['ci_low'], row['ci_high'], color='#0F6F73', lw=2)
                    ax.scatter(row['estimate'], y, color='#0F6F73', s=35, zorder=3)
                else: ax.scatter(row['estimate'], y, facecolors='none', edgecolors='#BD6F20', s=40, lw=1.5, zorder=3)
            labels = [textwrap.fill(row['run'] + (f' | {row["model"]}' if row['model'] != row['run'] else '') + (f' | seed {row["seed"]}' if row['seed'] else ''), width=60) for _, row in chunk.iterrows()]
            ax.set_yticks(ys, labels); ax.invert_yaxis(); metric = context['metric']
            ax.set_xlabel(metric.upper() if metric in ['r2', 'mae', 'rmse', 'auc'] else metric.replace('_', ' '))
            if metric == 'r2' or context['result_scope'] == 'seed_paired_effect': ax.axvline(0, color='#899499', lw=.8, linestyle='--')
            if metric in ['auc', 'average_precision', 'accuracy', 'balanced_accuracy']: ax.set_xlim(0, 1)
            ax.set_title(f'{context["target"]} | {context["cohort"]} | {context["gender"]} | {context["platform"]}\n{context["evaluation_split"]} | {context["metric_scale"]} | {context["resampling_unit"]} intervals')
            short_root = '/'.join(normalized_path(context['source_root']).split('/')[-3:])
            fig.text(.01, .01, textwrap.fill(f'Filled: actual 95% CI. Hollow: no 95% CI. {context["evidence_level"]}. Root: {short_root}', width=125), fontsize=8)
            digest = hashlib.sha256('|'.join(map(str, key)).encode()).hexdigest()[:12]; stem = f'{metric}_{digest}_{page}'
            fig.tight_layout(rect=(0, .10, 1, 1))
            fig.savefig(OUTPUT / f'{stem}.png', dpi=160, bbox_inches='tight'); fig.savefig(OUTPUT / f'{stem}.pdf', bbox_inches='tight')
            plot_manifest.append({**context, 'rows': len(chunk), 'png': str(OUTPUT / f'{stem}.png'), 'pdf': str(OUTPUT / f'{stem}.pdf')})
            plt.show(); plt.close(fig)
pd.DataFrame(plot_manifest).to_csv(OUTPUT / 'performance_plot_manifest.csv', index=False)
print(f'Saved {len(plot_manifest)} performance panels to {OUTPUT}.')

In [ ]:
TOTAL_TARGETS = {'peripheral_total', 'volume_peripheral_total', 'plaque_volume_total_corrected'}
total = tidy.loc[tidy['target'].isin(TOTAL_TARGETS)].copy() if not tidy.empty else pd.DataFrame()
if not total.empty:
    total.to_csv(OUTPUT / 'total_peripheral_volume_metrics.csv', index=False)
    display(total[['platform', 'source_root', 'run', 'model', 'cohort', 'gender', 'seed', 'metric', 'estimate', 'ci_low', 'ci_high', 'ci_status', 'evaluation_split']])
else: print('No explicitly identified total peripheral volume result; no coronary-inclusive sum is inferred.')
if not tidy.empty:
    neural = tidy['model'].eq('meshvae') | tidy['model'].eq(tidy['run']) | tidy['model'].str.contains('boson', case=False)
    matched = tidy['run'].str.contains(r'all_log_(?:gaussian_mmd|boson_mmd)', case=False, regex=True) & neural
    boson = tidy.loc[matched | (tidy['prior'].str.contains('boson', case=False) & neural) | tidy['model'].str.contains('boson', case=False) | tidy['result_scope'].str.startswith('seed_')]
    boson.to_csv(OUTPUT / 'boson_metrics.csv', index=False)
    if not boson.empty: display(boson[['platform', 'run', 'model', 'target', 'gender', 'seed', 'result_scope', 'metric', 'estimate', 'resampling_unit', 'has_95ci']])
    else: print('No saved Boson results identified.')
    paired = tidy.loc[matched & tidy['result_scope'].eq('model_evaluation')].copy()
    if not paired.empty:
        def verified_hashes(value):
            try:
                hashes = json.loads(value)
                return isinstance(hashes, dict) and bool(hashes) and all(str(item).strip() for item in hashes.values())
            except (TypeError, json.JSONDecodeError): return False
        verified = paired['reference_fingerprint'].str.strip().ne('') & paired['reference_split_hashes'].map(verified_hashes)
        paired.loc[~verified].to_csv(OUTPUT / 'boson_pairing_unverified.csv', index=False)
        if not verified.all(): print('Boson runs without a saved reference fingerprint and split hashes are shown individually; no automatic paired effect.')
        paired = paired.loc[verified].copy()
        paired['arm'] = np.where(paired['run'].str.contains('all_log_boson_mmd', case=False), 'boson_mmd', 'gaussian_mmd')
        keys = ['platform', 'source_root', 'target', 'cohort', 'gender', 'seed', 'metric', 'metric_scale', 'reference_fingerprint', 'reference_split_hashes']
        if paired.duplicated(keys + ['arm']).any():
            print('Multiple matched candidates for a seed/context; no automatic paired effect calculated.')
            paired.to_csv(OUTPUT / 'boson_pairing_ambiguous.csv', index=False)
        else:
            comparison = paired.pivot(index=keys, columns='arm', values='estimate').reset_index()
            if {'boson_mmd', 'gaussian_mmd'}.issubset(comparison):
                comparison['boson_minus_gaussian'] = comparison['boson_mmd'] - comparison['gaussian_mmd']
                comparison.to_csv(OUTPUT / 'boson_matched_seed_differences.csv', index=False)
                display(comparison)
                print('Per-seed differences; participant CIs are in the plots. No seed-level CI is inferred from these differences.')

## 4. t-SNE of selected checkpoints
Export latents on the server once with existing `extract_mesh_latents.py` (a forward pass, not retraining), then collect again with `--include-plot-data`. Each checkpoint and sex is plotted separately. Scaling, PCA and k-means are training-fitted. Train and test participants share one descriptive embedding. Statistical CIs above belong to performance metrics; t-SNE dots do not receive confidence bars.

Volume colors are measured `log1p(volume)`, retaining zeros. Existing latent merge requires plaque status, age and BMI and records complete-case counts. This may exclude participants. If conditioning inputs are not documented, the figure says **inputs unverified** rather than implying a scan-only representation. Existing coordinate files can also be plotted without a new t-SNE fit.

In [ ]:
latent_jobs = list(LATENT_JOBS)
latent_status = []
if not artifacts.empty and {'source_file', 'exported_file'}.issubset(artifacts):
    for _, a in artifacts.loc[artifacts['source_file'].map(normalized_path).str.endswith('/latent_manifest.json')].iterrows():
        try:
            manifest = json.loads(artifact_path(a).read_text(encoding='utf-8'))
            folder = normalized_path(a['source_file']).rsplit('/', 1)[0]
            # An older latent manifest may omit its training target. Recover it from
            # the owning study, so a specific territory never gets generic any-plaque colors.
            study_original = folder.split('/study/', 1)[0] + '/study/manifest.json' if '/study/' in folder else normalized_path(a.get('run_directory', '')) + '/study/manifest.json'
            study_path = local_artifact(study_original, a['_export_dir'])
            study = json.loads(study_path.read_text(encoding='utf-8')) if study_path else {}
            territory = study.get('experiment', {}).get('territory')
            target_col = manifest.get('target_col') or ('presence_' + territory if territory else None)
            volume_col = manifest.get('volume_col') or ('volume_' + territory if territory else None)
            latent_jobs.append(dict(name=manifest['experiment'], gender=manifest['gender'],
                representation_label=manifest.get('representation_label') or f'{manifest["experiment"]}; posterior mean; inputs unverified',
                train_latent=folder + '/train_latent_embeddings.csv', test_latent=folder + '/test_latent_embeddings.csv',
                train_tsv=manifest['train_tsv'], test_tsv=manifest['test_tsv'], target_col=target_col, volume_col=volume_col,
                id_col=manifest.get('id_col'), age_col=manifest.get('age_col'), gender_col=manifest.get('gender_col'),
                bmi_col=manifest.get('bmi_col'), export_dir=a['_export_dir']))
        except Exception as error: latent_status.append(dict(name=a['source_file'], status='bad_manifest', reason=str(error)))
seen_jobs = set()
for job in latent_jobs:
    identity = (job['name'], job['gender'], str(job['train_latent']), str(job['test_latent']))
    if identity in seen_jobs: continue
    seen_jobs.add(identity); status = {'name': job['name'], 'gender': job['gender'], 'representation': job.get('representation_label', 'Inputs unverified')}
    if not RUN_TSNE: latent_status.append({**status, 'status': 'disabled'}); continue
    resolved = {key: local_artifact(job[key], job.get('export_dir')) for key in ['train_latent', 'test_latent', 'train_tsv', 'test_tsv']}
    absent = [key for key, path in resolved.items() if path is None]
    if absent: latent_status.append({**status, 'status': 'missing_inputs', 'reason': ', '.join(absent)}); continue
    try:
        train_latent, train_cols = load_latent(resolved['train_latent'], 'train', job.get('id_col'))
        test_latent, test_cols = load_latent(resolved['test_latent'], 'test', job.get('id_col'))
        columns = sorted(set(train_cols) & set(test_cols))
        if not columns: raise ValueError('No matching latent dimensions.')
        if set(train_latent.participant_id) & set(test_latent.participant_id): raise ValueError('Train/test participant overlap.')
        frames, flows = [], []
        source_columns = read_table(resolved['train_tsv']).columns
        if not job.get('id_col') and 'participant_id' in source_columns: job['id_col'] = 'participant_id'
        if not job.get('age_col') and 'study_age' in source_columns: job['age_col'] = 'study_age'
        if not job.get('gender_col') and 'study_gender' in source_columns: job['gender_col'] = 'study_gender'
        if not job.get('target_col'):
            job['target_col'] = 'presence_peripheral_total' if 'presence_peripheral_total' in source_columns else 'anyplaq'
        for split, latent in [('train', train_latent), ('test', test_latent)]:
            metadata, _ = load_metadata(resolved[f'{split}_tsv'], job['target_col'], job.get('id_col'), job.get('age_col'), job.get('bmi_col'), job.get('gender_col'))
            source = read_table(resolved[f'{split}_tsv'])
            volume_col = job.get('volume_col') or next((c for c in ['volume_peripheral_total', 'plaque_volume_total_corrected', 'peripheral_total'] if c in source), None)
            if volume_col:
                if volume_col not in source: raise KeyError(f'Missing measured volume column {volume_col!r}.')
                colors = pd.DataFrame({'participant_id': source[resolve_id_column(source, job.get('id_col'))].map(clean_participant_id), 'plaque_volume': numeric(source[volume_col])})
                colors['plaque_volume'] = colors['plaque_volume'].where(np.isfinite(colors['plaque_volume']) & colors['plaque_volume'].ge(0))
                metadata = metadata.merge(colors, on='participant_id', how='left', validate='one_to_one')
            frame, flow = merge_split(metadata, latent[['participant_id', *columns]], columns, split)
            frame = frame.loc[frame['gender'].eq(job['gender'])].reset_index(drop=True); flow['selected_gender_rows'] = len(frame)
            frames.append(frame); flows.append(flow)
        train, test = frames
        if len(train) < 4 or len(test) < 3 or len(train) + len(test) < 8: raise ValueError('Too few complete participants for t-SNE.')
        fingerprint = hashlib.sha256('|'.join(map(str, identity)).encode() + b''.join(path.read_bytes() for path in resolved.values()) + f'{SEED}|{PERPLEXITY}'.encode()).hexdigest()[:12]
        folder = OUTPUT / 'tsne' / (re.sub(r'[^A-Za-z0-9_.-]+', '_', job['name']) + '_' + job['gender'] + '_' + fingerprint)
        folder.mkdir(parents=True, exist_ok=True); pd.DataFrame(flows).to_csv(folder / 'cohort_flow.csv', index=False)
        if not (folder / 'tsne_latent_signal.png').exists():
            train_values, test_values, info = standardize_latent(train, test, columns, SEED)
            kmeans, scores, k = choose_kmeans(train_values, 2, 8, SEED)
            info.update(make_tsne_figure(train_values, test_values, train, test, kmeans.predict(train_values), kmeans.predict(test_values), folder, SEED, PERPLEXITY, job['gender'], job.get('representation_label', 'Inputs unverified')))
            (folder / 'analysis_manifest.json').write_text(json.dumps({**job, **info, 'note': 'Descriptive embedding; training-fitted scaler/PCA/clusters.'}, indent=2, default=str), encoding='utf-8')
        display(Image(filename=str(folder / 'tsne_latent_signal.png')))
        latent_status.append({**status, 'status': 'completed', 'output': str(folder), 'n_train': len(train), 'n_test': len(test), 'volume_panel': 'plaque_volume' in train})
    except Exception as error: latent_status.append({**status, 'status': 'failed', 'reason': f'{type(error).__name__}: {error}'})
if RUN_TSNE and not artifacts.empty and 'source_file' in artifacts:
    for _, a in artifacts.loc[artifacts['source_file'].str.endswith('tsne_coordinates.csv')].iterrows():
        try:
            frame = pd.read_csv(artifact_path(a))
            if not {'tsne_1', 'tsne_2', 'target'}.issubset(frame): raise ValueError('Coordinates need tsne_1, tsne_2 and measured target.')
            fig, axes = plt.subplots(1, 2 if 'plaque_volume' in frame else 1, figsize=(11 if 'plaque_volume' in frame else 6, 4.5), squeeze=False); axes = axes[0]
            splits = frame.get('split', pd.Series('unspecified', index=frame.index))
            for split in splits.unique():
                mask = splits.eq(split)
                axes[0].scatter(frame.loc[mask, 'tsne_1'], frame.loc[mask, 'tsne_2'], c=frame.loc[mask, 'target'], cmap='coolwarm', vmin=0, vmax=1, marker='^' if split == 'test' else 'o', s=13, alpha=.6, label=str(split))
            axes[0].set_title('Measured plaque status: blue 0 / red 1'); axes[0].legend(frameon=False)
            if len(axes) > 1:
                valid = np.isfinite(frame['plaque_volume']) & frame['plaque_volume'].ge(0)
                points = axes[1].scatter(frame.loc[valid, 'tsne_1'], frame.loc[valid, 'tsne_2'], c=np.log1p(frame.loc[valid, 'plaque_volume']), cmap='magma', s=13, alpha=.6)
                fig.colorbar(points, ax=axes[1], label='log1p(measured plaque volume in reported units)'); axes[1].set_title('Measured volume')
            for axis in axes: axis.set_xticks([]); axis.set_yticks([]); axis.set_xlabel('t-SNE 1'); axis.set_ylabel('t-SNE 2')
            fig.suptitle('Previously saved coordinates: ' + normalized_path(a['source_file']), fontsize=8)
            folder = OUTPUT / 'tsne'; folder.mkdir(exist_ok=True); digest = hashlib.sha256(a['source_file'].encode()).hexdigest()[:12]
            fig.tight_layout(); fig.savefig(folder / f'existing_{digest}.png', dpi=160, bbox_inches='tight'); fig.savefig(folder / f'existing_{digest}.pdf', bbox_inches='tight')
            plt.show(); plt.close(fig)
            latent_status.append(dict(name=a['source_file'], status='saved_coordinates', output=str(folder / f'existing_{digest}.png')))
        except Exception as error: latent_status.append(dict(name=a['source_file'], status='failed', reason=str(error)))
pd.DataFrame(latent_status, columns=sorted(set().union(*(row.keys() for row in latent_status))) if latent_status else ['name', 'status', 'reason']).to_csv(OUTPUT / 'latent_input_inventory.csv', index=False)
if latent_status: display(pd.DataFrame(latent_status))
else: print('No latent manifests or t-SNE coordinates found. Export selected checkpoints with extract_mesh_latents.py, then collect with --include-plot-data.')

## Outputs
PNG/PDF performance panels and t-SNE figures are saved beside `tidy_metrics.csv`, total peripheral-volume and Boson tables, and missing/failed-input inventories. `performance_plot_manifest.csv` maps every plot to its full evaluation context.

If a model is missing, inspect `source_inventory.csv`: a checkpoint without saved diagnostics is an unevaluated model, not a zero score. Missing error bars are explained in `ci_computation_inventory.csv` and `missing_confidence_intervals.csv`. No error bars can be reconstructed from screenshot summaries.

In [ ]:
display(pd.DataFrame({'output': ['tidy_metrics.csv', 'performance_plot_manifest.csv', 'missing_confidence_intervals.csv', 'ci_computation_inventory.csv', 'latent_input_inventory.csv'], 'folder': [str(OUTPUT)] * 5}))
(OUTPUT / 'notebook_manifest.json').write_text(json.dumps({'exports': [str(path) for path in EXPORT_DIRS], 'bootstrap_replicates': BOOTSTRAPS, 'confidence': CONFIDENCE, 'seed': SEED, 'performance_panels': len(plot_manifest), 'latent_jobs': len(latent_jobs), 'note': 'Original provenance preserved; missing intervals explicit; no training.'}, indent=2), encoding='utf-8')